In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q rasterio geopandas shapely

Mounted at /content/drive


In [ ]:
import os

mask_path  = "/content/drive/MyDrive/treesUIS.tif"   # EDIT ME — your SAM3 tree mask
output_dir = "/content/drive/MyDrive/tree_placement_output"  # EDIT ME
os.makedirs(output_dir, exist_ok=True)

# --- The parameter that matters most: minimum spacing between trees, in METERS ---
# This is a proxy for average crown diameter / how tightly trees can be packed.
# Tune this to your campus: widely-spaced ornamental trees -> larger value (e.g. 4-6m).
# Denser plantings / hedgerows -> smaller value (e.g. 2-3m).
min_spacing_m = 7

# Optional: cap on candidate points tried per active point before giving up on it
# (standard Bridson parameter, rarely needs changing)
k_candidates = 30

# Optional random seed for reproducibility
random_seed = 42

# --- New: keep generated points away from the ragged edges of each mask blob ---
# Mask boundaries from SAM3 are often noisy pixel-level artifacts, not real
# canopy edges, so this shrinks the "allowed" sampling area inward by this
# many meters from every mask boundary. Set to 0 to disable.
edge_margin_m = 1.0

# --- New: ignore mask regions smaller than this size (likely false positives,
# e.g. small bushes, mask noise, or fragments too small to be a real tree). ---
# Measured as the longest side of each connected region's bounding box, in meters.
# Set to 0 to disable.
min_region_length_m = 1.0

In [ ]:
import numpy as np
import rasterio

with rasterio.open(mask_path) as src:
    mask = src.read(1) > 0          # boolean array, True = tree zone
    transform = src.transform
    crs = src.crs
    px_size_x = transform[0]
    px_size_y = -transform[4]        # transform[4] is negative for north-up rasters

print(f"Mask shape: {mask.shape}, pixel size: {px_size_x:.3f} x {px_size_y:.3f} m")
print(f"Tree-covered fraction: {mask.mean():.3f}")

# %% CELL 3b — Filter small regions, then erode edges inward (margin)
from scipy import ndimage as ndi

# --- Step 1: drop connected regions smaller than min_region_length_m ---
mask_filtered = mask.copy()
if min_region_length_m > 0:
    labeled_orig, n_orig = ndi.label(mask)
    objects = ndi.find_objects(labeled_orig)
    n_dropped = 0
    for label_id, slc in enumerate(objects, start=1):
        if slc is None:
            continue
        region_h_px = slc[0].stop - slc[0].start
        region_w_px = slc[1].stop - slc[1].start
        extent_m = max(region_h_px * px_size_y, region_w_px * px_size_x)
        if extent_m < min_region_length_m:
            mask_filtered[labeled_orig == label_id] = False
            n_dropped += 1
    print(f"Dropped {n_dropped}/{n_orig} regions smaller than {min_region_length_m}m")

# --- Step 2: erode inward by edge_margin_m so points avoid ragged mask edges ---
if edge_margin_m > 0:
    dist_from_edge_px = ndi.distance_transform_edt(mask_filtered)
    margin_px = edge_margin_m / ((px_size_x + px_size_y) / 2)
    sampling_mask = mask_filtered & (dist_from_edge_px >= margin_px)
else:
    sampling_mask = mask_filtered

print(f"Sampling area after filtering + margin: {sampling_mask.mean():.4f} "
      f"(was {mask.mean():.4f} before)")

Mask shape: (8634, 15638), pixel size: 0.056 x 0.056 m
Tree-covered fraction: 0.176
Dropped 752/1180 regions smaller than 1.0m
Sampling area after filtering + margin: 0.1295 (was 0.1758 before)


In [ ]:
import random
import math

random.seed(random_seed)
np.random.seed(random_seed)

height_px, width_px = mask.shape
width_m = width_px * px_size_x
height_m = height_px * px_size_y

r = min_spacing_m
cell_size = r / math.sqrt(2)
grid_w = int(width_m / cell_size) + 1
grid_h = int(height_m / cell_size) + 1
grid = -np.ones((grid_h, grid_w), dtype=int)   # stores index into samples list, -1 = empty

samples = []       # list of (x_m, y_m) accepted points, in meters from raster origin (top-left)
active_list = []

def world_to_pixel(x_m, y_m):
    col = int(x_m / px_size_x)
    row = int(y_m / px_size_y)
    return row, col

def is_in_mask(x_m, y_m):
    row, col = world_to_pixel(x_m, y_m)
    if row < 0 or row >= height_px or col < 0 or col >= width_px:
        return False
    return bool(sampling_mask[row, col])

def grid_coords(x_m, y_m):
    return int(y_m / cell_size), int(x_m / cell_size)

def fits(x_m, y_m):
    if not is_in_mask(x_m, y_m):
        return False
    gy, gx = grid_coords(x_m, y_m)
    for dy in range(-2, 3):
        for dx in range(-2, 3):
            ny, nx = gy + dy, gx + dx
            if 0 <= ny < grid_h and 0 <= nx < grid_w:
                idx = grid[ny, nx]
                if idx != -1:
                    ox, oy = samples[idx]
                    if (ox - x_m) ** 2 + (oy - y_m) ** 2 < r ** 2:
                        return False
    return True

def add_sample(x_m, y_m):
    idx = len(samples)
    samples.append((x_m, y_m))
    active_list.append(idx)
    gy, gx = grid_coords(x_m, y_m)
    grid[gy, gx] = idx

# --- Seed initial points: scan for centroids per connected region of the
# FINAL sampling_mask (post filtering + erosion) so we don't rely on one
# random starting point, and so thin regions that vanished under erosion
# are correctly skipped rather than seeding a point outside the allowed area.
labeled, n_labels = ndi.label(sampling_mask)
print(f"Found {n_labels} connected regions (post filter+margin) to seed sampling from")

for label_id in range(1, n_labels + 1):
    ys, xs = np.where(labeled == label_id)
    if len(ys) == 0:
        continue
    cy_px, cx_px = ys.mean(), xs.mean()
    x_m = cx_px * px_size_x
    y_m = cy_px * px_size_y
    # centroid of an irregular region can itself fall outside the region
    # (e.g. a crescent shape) -- fall back to the nearest actual True pixel
    if not is_in_mask(x_m, y_m):
        dists = (ys - cy_px) ** 2 + (xs - cx_px) ** 2
        nearest = np.argmin(dists)
        cy_px, cx_px = ys[nearest], xs[nearest]
        x_m = cx_px * px_size_x
        y_m = cy_px * px_size_y
    if fits(x_m, y_m):
        add_sample(x_m, y_m)

# --- Bridson's algorithm main loop ---
while active_list:
    i = random.choice(active_list)
    ox, oy = samples[i]
    found = False
    for _ in range(k_candidates):
        angle = random.uniform(0, 2 * math.pi)
        radius = random.uniform(r, 2 * r)
        nx_m = ox + radius * math.cos(angle)
        ny_m = oy + radius * math.sin(angle)
        if fits(nx_m, ny_m):
            add_sample(nx_m, ny_m)
            found = True
    if not found:
        active_list.remove(i)

print(f"\nGenerated {len(samples)} candidate tree positions")

Found 400 connected regions (post filter+margin) to seed sampling from

Generated 1160 candidate tree positions


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

geo_points = []
for x_m, y_m in samples:
    row, col = world_to_pixel(x_m, y_m)
    world_x, world_y = rasterio.transform.xy(transform, row, col)
    geo_points.append(Point(world_x, world_y))

gdf = gpd.GeoDataFrame({"tree_id": range(1, len(geo_points) + 1)}, geometry=geo_points, crs=crs)

# Add lat/lon columns too, useful if your 3D pipeline wants geographic coords
if not gdf.crs.is_geographic:
    gdf_wgs84 = gdf.to_crs(4326)
    gdf["lon"] = gdf_wgs84.geometry.x
    gdf["lat"] = gdf_wgs84.geometry.y

gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

gpkg_path = os.path.join(output_dir, "tree_positions.gpkg")
csv_path  = os.path.join(output_dir, "tree_positions.csv")

gdf.to_file(gpkg_path, driver="GPKG")
gdf.drop(columns="geometry").to_csv(csv_path, index=False)

print(f"Saved {len(gdf)} tree positions to:\n  {gpkg_path}\n  {csv_path}")

Saved 1160 tree positions to:
  /content/drive/MyDrive/tree_placement_output/tree_positions.gpkg
  /content/drive/MyDrive/tree_placement_output/tree_positions.csv
